In [ ]:
!mkdir -p megha data checkpoints
!pip install transformers tokenizers accelerate huggingface_hub


In [ ]:
%%writefile megha/__init__.py



In [ ]:
%%writefile megha/config.py
from dataclasses import dataclass

@dataclass
class MeghaConfig:
    vocab_size: int = 8000        # 8k ByteLevel vocabulary
    max_seq_len: int = 256
    d_model: int = 384            # 17.37M params
    n_layers: int = 8             # 8 Transformer blocks
    n_heads: int = 6              # 6 heads (64 dim per head)
    dropout: float = 0.1
    batch_size: int = 4           # Small batch for gradient stability & higher step resolution
    grad_accum_steps: int = 2     # Effective batch size = 8
    learning_rate: float = 3e-4
    epochs: int = 25              # 25 epochs (~3,500+ steps) for full convergence
    warmup_steps: int = 150       # Linear LR Warmup steps



In [ ]:
%%writefile megha/model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from .config import MeghaConfig

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        var = torch.mean(x ** 2, dim=-1, keepdim=True)
        return x * torch.rsqrt(var + self.eps) * self.weight

class SwiGLU(nn.Module):
    def __init__(self, d_model: int, hidden_dim: int, dropout: float = 0.1):
        super().__init__()
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, d_model, bias=False)
        self.w3 = nn.Linear(d_model, hidden_dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))

class MeghaAttention(nn.Module):
    def __init__(self, config: MeghaConfig):
        super().__init__()
        self.n_heads = config.n_heads
        self.head_dim = config.d_model // config.n_heads
        self.d_model = config.d_model
        
        self.qkv = nn.Linear(config.d_model, 3 * config.d_model, bias=False)
        self.out_proj = nn.Linear(config.d_model, config.d_model, bias=False)
        self.dropout_p = config.dropout

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        # Fast PyTorch Scaled Dot Product Attention with automatic Causal Mask
        out = F.scaled_dot_product_attention(
            q, k, v, 
            is_causal=True, 
            dropout_p=self.dropout_p if self.training else 0.0
        )
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.out_proj(out)

class MeghaBlock(nn.Module):
    def __init__(self, config: MeghaConfig):
        super().__init__()
        self.norm1 = RMSNorm(config.d_model)
        self.attn = MeghaAttention(config)
        self.norm2 = RMSNorm(config.d_model)
        hidden_dim = int(4 * config.d_model * 2 / 3)   # SwiGLU standard dimension scaling
        self.mlp = SwiGLU(config.d_model, hidden_dim, dropout=config.dropout)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class MeghaModel(nn.Module):
    def __init__(self, config: MeghaConfig):
        super().__init__()
        self.config = config
        
        self.token_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_emb = nn.Embedding(config.max_seq_len, config.d_model)
        
        self.blocks = nn.ModuleList([MeghaBlock(config) for _ in range(config.n_layers)])
        self.norm_f = RMSNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        
        # Weight tying
        self.token_emb.weight = self.lm_head.weight
        
        self.apply(self._init_weights)
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        
        x = self.token_emb(idx) + self.pos_emb(pos)
        for block in self.blocks:
            x = block(x)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            # Ignore -100 index so loss is ONLY calculated on answer tokens!
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(logits.view(-1, self.config.vocab_size), targets.view(-1))
            
        return logits, loss
        
    def generate(self, idx, max_new_tokens, temperature=0.7, top_k=40, eos_token_id=None, repetition_penalty=1.2):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)

            if repetition_penalty != 1.0:
                for token_id in set(idx[0].tolist()):
                    if logits[0, token_id] > 0:
                        logits[0, token_id] /= repetition_penalty
                    else:
                        logits[0, token_id] *= repetition_penalty
            
            if top_k is not None and top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
                
            probs = torch.nn.functional.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

            if eos_token_id is not None and idx_next.item() == eos_token_id:
                break

        return idx



In [ ]:
%%writefile megha/tokenizer.py
import os
import json
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from .config import MeghaConfig

class MeghaTokenizer:
    def __init__(self, config: MeghaConfig):
        self.config = config
        self.tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
        self.tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
        self.tokenizer.decoder = ByteLevelDecoder()
        self.trainer = BpeTrainer(
            vocab_size=config.vocab_size,
            special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]", "<|endoftext|>"]
        )
        
    def train_from_iterator(self, iterator):
        self.tokenizer.train_from_iterator(iterator, self.trainer)
        
    def save(self, path):
        self.tokenizer.save(path)
        
    def load(self, path):
        self.tokenizer = Tokenizer.from_file(path)
        
    def encode(self, text):
        return self.tokenizer.encode(text).ids
        
    def decode(self, ids):
        return self.tokenizer.decode(ids)

    def get_eos_token_id(self):
        """Return the dedicated token ID for <|endoftext|>."""
        vocab = self.tokenizer.get_vocab()
        return vocab.get("<|endoftext|>", 0)

if __name__ == "__main__":
    # Script to train the tokenizer on generated curriculum data
    print("Training Tokenizer...")
    config = MeghaConfig()
    megha_tok = MeghaTokenizer(config)
    
    # 1. Extract text into an iterator from all available curriculum files
    def text_iterator():
        import glob
        files = glob.glob("data/level_*_curriculum.json")
        for file_path in files:
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    dataset = json.load(f)
                    for item in dataset:
                        if isinstance(item, dict) and "text" in item and item["text"]:
                            yield item["text"]
            except Exception:
                pass
            
    # 2. Train
    megha_tok.train_from_iterator(text_iterator())
    
    # 4. Save
    os.makedirs("data", exist_ok=True)
    megha_tok.save("data/tokenizer.json")
    print(f"Tokenizer trained and saved to data/tokenizer.json with vocab size: {megha_tok.tokenizer.get_vocab_size()}")
    
    # Quick Test
    test_text = "The computer is on."
    encoded = megha_tok.encode(test_text)
    print(f"\nTest string: '{test_text}'")
    print(f"Encoded IDs: {encoded}")
    print(f"Decoded: '{megha_tok.decode(encoded)}'")



In [ ]:
%%writefile megha/dataset.py
import json
import glob
import torch
import random
from torch.utils.data import Dataset, DataLoader
from .tokenizer import MeghaTokenizer
from .config import MeghaConfig
import os

class MeghaDataset(Dataset):
    def __init__(self, all_texts: list, tokenizer: MeghaTokenizer, config: MeghaConfig):
        self.config = config
        self.tokenizer = tokenizer
        
        vocab = tokenizer.tokenizer.get_vocab()
        eos_id = vocab.get("<|endoftext|>", 0)
        
        all_x_tokens = []
        all_y_tokens = []
        
        for text in all_texts:
            if not text or "Q:" not in text or "A:" not in text:
                continue
            parts = text.split("A:", 1)
            prompt_str = parts[0] + "A:"
            answer_str = parts[1]
            
            prompt_ids = self.tokenizer.encode(prompt_str)
            answer_ids = self.tokenizer.encode(answer_str)
            if not answer_ids:
                continue
            answer_ids.append(eos_id)
            
            # Input: prompt + answer
            x_seq = prompt_ids + answer_ids
            # Target: -100 for prompt tokens (no loss gradient), answer_ids for answer tokens
            y_seq = [-100] * len(prompt_ids) + answer_ids
            
            all_x_tokens.extend(x_seq)
            all_y_tokens.extend(y_seq)
            
        # Pad with EOS / -100 if sequence is short
        min_len = self.config.max_seq_len + 1
        while len(all_x_tokens) < min_len:
            all_x_tokens.extend([eos_id] * 20)
            all_y_tokens.extend([-100] * 20)
            
        self.x_data = torch.tensor(all_x_tokens, dtype=torch.long)
        self.y_data = torch.tensor(all_y_tokens, dtype=torch.long)
        
        self.stride = max(1, self.config.max_seq_len // 2)
        
    def __len__(self):
        return max(1, (len(self.x_data) - self.config.max_seq_len - 1) // self.stride)
        
    def __getitem__(self, idx):
        start_idx = idx * self.stride
        x = self.x_data[start_idx : start_idx + self.config.max_seq_len]
        # Shift target by 1 token for standard next-token prediction
        y = self.y_data[start_idx + 1 : start_idx + self.config.max_seq_len + 1]
        return x, y


def load_texts_from_file(data_path: str) -> list:
    """Load all Q&A texts from a single curriculum JSON file."""
    if not os.path.exists(data_path):
        return []
    try:
        with open(data_path, "r", encoding="utf-8") as f:
            raw_data = json.load(f)
        texts = []
        for item in raw_data:
            text = item.get("text", "")
            if text and len(text.strip()) > 5:
                texts.append(text.strip())
        return texts
    except Exception as e:
        print(f"Warning: Could not load {data_path}: {e}")
        return []


def get_combined_dataloader(tokenizer_path: str, config: MeghaConfig, data_dir: str = "data"):
    """
    MIXED TRAINING: Load ALL levels' data, shuffle together, train ONE model.
    This prevents Catastrophic Forgetting.
    """
    tokenizer = MeghaTokenizer(config)
    if os.path.exists(tokenizer_path):
        tokenizer.load(tokenizer_path)
    else:
        raise FileNotFoundError(f"Tokenizer not found at {tokenizer_path}. Run tokenizer.py first.")
    
    # Load all curriculum files
    all_texts = []
    files = sorted(glob.glob(f"{data_dir}/level_*_curriculum.json"))
    
    for fpath in files:
        texts = load_texts_from_file(fpath)
        all_texts.extend(texts)
        print(f"  Loaded {len(texts)} examples from {os.path.basename(fpath)}")
    
    if not all_texts:
        raise ValueError("No training data found! Run data_gen.py first.")
    
    # SHUFFLE: mix all levels together so model learns all topics uniformly
    random.shuffle(all_texts)
    print(f"\nTotal training examples (all levels combined): {len(all_texts)}")
    
    dataset = MeghaDataset(all_texts, tokenizer, config)
    print(f"Total dataset chunks: {len(dataset)}")
    
    dataloader = DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=True,
        drop_last=False
    )
    return dataloader, tokenizer


def get_dataloader(data_path: str, tokenizer_path: str, config: MeghaConfig):
    """Single-level loader (kept for backward compatibility)."""
    tokenizer = MeghaTokenizer(config)
    if os.path.exists(tokenizer_path):
        tokenizer.load(tokenizer_path)
    else:
        raise FileNotFoundError(f"Tokenizer not found at {tokenizer_path}. Run tokenizer.py first.")
    
    texts = load_texts_from_file(data_path)
    dataset = MeghaDataset(texts, tokenizer, config)
    dataloader = DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=True,
        drop_last=False
    )
    return dataloader, tokenizer



In [ ]:
%%writefile megha/data_gen.py
import json
import argparse
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Prompts for different levels of curriculum
LEVEL_PROMPTS = {
    0: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 0 - Basic English Grammar and Vocabulary.
Generate 50 simple Q&A examples covering basic sentence structure, nouns, verbs, and pronouns.
Format each example STRICTLY as: "Q: <simple question>\nA: <simple answer>".
Format the output STRICTLY as a JSON array of objects, each with a "text" field.
Example: [{"text": "Q: Is the cat sleeping?\nA: Yes, the cat is sleeping on the bed."}]
Output nothing but the JSON array. Do not include markdown blocks.""",
    
    1: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 1 - General Knowledge & Basic Reasoning.
Generate 50 Q&A examples covering numbers, comparison, time, input/output, and basic cause-effect reasoning.
Format each example STRICTLY as: "Q: <question>\nA: <clear answer>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    2: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 2 - Computer Fundamentals.
Generate 50 Q&A examples covering CPU, RAM, Storage (HDD vs SSD), Operating Systems (kernel, processes), and basic computing.
Format each example STRICTLY as: "Q: <question>\nA: <clear factual answer>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    3: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 3 - Linux Operating System.
Generate 50 Q&A examples covering Linux commands (chmod, chown, ls, grep, ps, systemctl), filesystem (/etc, /var), and file permissions.
Format each example STRICTLY as: "Q: <question>\nA: <clear factual answer>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    4: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 4 - Networking.
Generate Q&A examples covering TCP/IP, OSI model layers, DNS, CIDR subnetting, HTTP status codes (200, 404, 502), and common ports (22, 80, 443).
Format each example STRICTLY as: "Q: <question>\nA: <clear factual answer>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    5: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 5 - Cloud Computing Fundamentals.
Generate Q&A examples covering virtualization, Cloud models (IaaS, PaaS, SaaS), deployment models (public, private, hybrid), and high availability.
Format each example STRICTLY as: "Q: <question>\nA: <clear factual answer>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    6: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 6 - AWS Core.
Generate Q&A examples covering Amazon EC2, S3 bucket storage, IAM roles and policies, VPC, and RDS databases.
Format each example STRICTLY as: "Q: <question>\nA: <clear factual answer>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    7: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 7 - Docker & Containers.
Generate Q&A examples covering Docker containers, images, Dockerfile instructions (FROM, RUN, CMD, COPY), docker build, docker run, and volumes.
Format each example STRICTLY as: "Q: <question>\nA: <clear factual answer>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    8: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 8 - Kubernetes.
Generate Q&A examples covering K8s pods, deployments, services (ClusterIP, NodePort), Ingress, and replica sets.
Format each example STRICTLY as: "Q: <question>\nA: <clear factual answer>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    9: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 9 - DevOps & CI/CD.
Generate Q&A examples covering Git commands, CI/CD pipelines, and Infrastructure as Code (Terraform).
Format each example STRICTLY as: "Q: <question>\nA: <clear factual answer>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    10: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 10 - Cloud Security.
Generate Q&A examples covering authentication, IAM policies, why public S3 buckets are dangerous, Zero Trust, and KMS encryption keys.
Format each example STRICTLY as: "Q: <question>\nA: <clear factual answer>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    11: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 11 - Cloud Troubleshooting.
Generate troubleshooting Q&A examples: symptoms, diagnosis, and fix (e.g. 502 Bad Gateway cause and fix, EC2 unreachable cause and fix, S3 AccessDenied).
Format each example STRICTLY as: "Q: <troubleshooting question>\nA: <clear diagnostic and resolution steps>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    12: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 12 - Cloud Architecture.
Generate Q&A examples covering Highly Available designs, Load Balancer + Auto Scaling, and Serverless API architectures.
Format each example STRICTLY as: "Q: <architectural question>\nA: <clear architectural design explanation>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    13: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 13 - Cloud Reasoning.
Generate scenario-based Q&A examples analyzing traffic spikes, failover strategies, and database bottlenecks.
Format each example STRICTLY as: "Q: <scenario question>\nA: <logical step-by-step reasoning and solution>".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    14: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 14 - CloudOps Multi-step Problem Solving.
Generate advanced Q&A examples showing step-by-step CloudOps problem resolution for Linux, AWS, Docker, and Kubernetes incidents.
Format each example STRICTLY as: "Q: <incident question>\nA: Identify symptoms -> Collect evidence -> Form hypothesis -> Test and Fix -> Verify.".
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks."""
}

def generate_curriculum_real(level: int):
    print(f"Loading Qwen model for Level {level} curriculum generation...")
    model_id = "Qwen/Qwen2.5-3B-Instruct"  
    
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, 
        torch_dtype=torch.float16,
        device_map="auto"
    )
    
    base_prompt = LEVEL_PROMPTS.get(level, LEVEL_PROMPTS[0])
    
    all_data = []
    target_examples = 250   # 250 Q&A pairs per level = ~3,750 total across 15 levels
    max_batches = 12
    
    print(f"Teacher is generating {target_examples} examples for Level {level} (in batches)...")
    
    sub_seeds = [
        "Focus on fundamental definitions, core concepts, and standard syntax.",
        "Focus on specific CLI flags, parameter configurations, and file paths.",
        "Focus on practical real-world scenarios, common pitfalls, and edge cases.",
        "Focus on security considerations, permissions, access controls, and best practices.",
        "Focus on error codes, log analysis, troubleshooting steps, and recovery.",
        "Focus on performance optimization, scaling, resource allocation, and automation."
    ]
    
    batch_count = 0
    while len(all_data) < target_examples and batch_count < max_batches:
        batch_count += 1
        sub_seed = sub_seeds[(batch_count - 1) % len(sub_seeds)]
        
        dynamic_prompt = f"{base_prompt}\n\nBatch {batch_count} instructions: {sub_seed}\nGenerate 30 unique Q&A examples. Output format MUST be:\nQ: <question>\nA: <answer>\n\nFormat all examples line-by-line as above."
        
        messages = [
            {"role": "system", "content": "You are an expert AI teacher generating curriculum dataset examples."},
            {"role": "user", "content": dynamic_prompt}
        ]
        
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        
        try:
            with torch.no_grad():
                generated_ids = model.generate(
                    **model_inputs,
                    max_new_tokens=2048,
                    temperature=0.85,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id
                )
            
            gen_tokens = [
                output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
            ]
            response = tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)[0].strip()
            
            # Primary parse: Regex extraction of Q: ... A: ... blocks
            import re
            qa_blocks = re.findall(r'(Q:\s*.*?\n\s*A:\s*.*?)(?=\n\s*Q:|\Z)', response, re.DOTALL)
            
            parsed_count = 0
            for block in qa_blocks:
                clean_block = block.strip()
                if "Q:" in clean_block and "A:" in clean_block and len(clean_block) >= 30:
                    all_data.append({"text": clean_block})
                    parsed_count += 1
                    
            # Secondary fallback: If regex found nothing, try JSON parse
            if parsed_count == 0:
                if "```json" in response:
                    response = response.split("```json")[1].split("```")[0]
                elif "```" in response:
                    response = response.split("```")[1].split("```")[0]
                try:
                    data = json.loads(response.strip(), strict=False)
                    if isinstance(data, list):
                        for d in data:
                            if isinstance(d, dict) and "text" in d:
                                all_data.append(d)
                except Exception:
                    pass
                    
            print(f"Batch {batch_count} generated {parsed_count} examples. Total for Level {level}: {len(all_data)}/{target_examples}")
            
        except Exception as e:
            print(f"Batch {batch_count} generation error: {e}")
            
    print(f"Successfully generated {len(all_data)} high-quality examples for Level {level}!")
    return all_data

def generate_curriculum_dummy(level: int):
    print(f"Generating DUMMY curriculum for Level {level} (Local PC Test)...")
    simulated_response = [
        {"text": "The computer is on."},
        {"text": "A network connects devices."},
        {"text": "She types on the keyboard."},
        {"text": "Data is stored in memory."},
        {"text": "He clicks the mouse."}
    ] * 100
    return simulated_response

def is_good_example(text: str) -> bool:
    """Filter out low-quality Q&A examples before training."""
    if not text or len(text.strip()) < 30:
        return False   # Too short — probably a parse fragment
    if len(text) > 900:
        return False   # Too long — probably multiple Q&As fused together
    if text.count("Q:") > 2:
        return False   # Multiple questions jammed in one entry
    if text.count("A:") > 2:
        return False   # Multiple answers jammed in one entry
    if not ("Q:" in text and "A:" in text):
        return False   # Not Q&A format at all
    return True

def save_curriculum(data, level):
    os.makedirs("data", exist_ok=True)

    # Apply quality filter
    before = len(data)
    data = [item for item in data if is_good_example(item.get("text", ""))]
    after = len(data)
    print(f"Quality filter: {before} → {after} examples ({before - after} removed)")

    file_path = f"data/level_{level}_curriculum.json"
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)
    print(f"Curriculum saved to {file_path} successfully!")

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Generate MEGHA Curriculum using Qwen")
    parser.add_argument("--level", type=int, default=0, help="Curriculum level to generate")
    parser.add_argument("--real", action="store_true", help="Use actual HuggingFace Qwen model (requires GPU)")
    args = parser.parse_args()
    
    if args.real:
        generated_data = generate_curriculum_real(args.level)
    else:
        generated_data = generate_curriculum_dummy(args.level)
        
    save_curriculum(generated_data, args.level)



In [ ]:
%%writefile megha/train.py
import torch
import torch.optim as optim
from .model import MeghaModel
from .config import MeghaConfig
from .dataset import get_combined_dataloader
import time
import os

def train_all():
    """
    MIXED TRAINING: Train ONE model on ALL 15 levels' data shuffled together.
    This eliminates Catastrophic Forgetting completely.
    """
    print("=" * 50)
    print("MEGHA MIXED TRAINING — All Levels Combined")
    print("=" * 50)
    
    config = MeghaConfig()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    model = MeghaModel(config).to(device)
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model Parameters: {num_params / 1e6:.2f} M\n")
    
    tokenizer_path = "data/tokenizer.json"
    
    try:
        dataloader, tokenizer = get_combined_dataloader(tokenizer_path, config)
    except (FileNotFoundError, ValueError) as e:
        print(f"ERROR: {e}")
        return
    
    total_batches = len(dataloader)
    if total_batches == 0:
        print("ERROR: Dataloader has 0 batches. Check your data files.")
        return
    
    grad_accum_steps = getattr(config, 'grad_accum_steps', 2)
    warmup_steps = getattr(config, 'warmup_steps', 150)
    total_steps = (total_batches // grad_accum_steps) * config.epochs
    
    print(f"\nBatches per epoch: {total_batches}")
    print(f"Gradient Accumulation Steps: {grad_accum_steps}")
    print(f"Epochs: {config.epochs}")
    print(f"Total optimization steps: {total_steps}")
    print(f"Warmup steps: {warmup_steps}\n")
    
    optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=0.01)
    
    def get_lr(step):
        if step < warmup_steps:
            return float(step + 1) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))
        
    import math
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr)
    
    model.train()
    global_step = 0
    optimizer.zero_grad()
    
    for epoch in range(config.epochs):
        print(f"\n--- Epoch {epoch+1}/{config.epochs} ---")
        epoch_loss = 0.0
        accum_loss = 0.0
        
        for step, (x, y) in enumerate(dataloader):
            t0 = time.time()
            x, y = x.to(device), y.to(device)
            
            logits, loss = model(x, targets=y)
            loss = loss / grad_accum_steps
            loss.backward()
            accum_loss += loss.item() * grad_accum_steps
            
            if (step + 1) % grad_accum_steps == 0 or (step + 1) == total_batches:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1
                
                dt = time.time() - t0
                if global_step % 20 == 0 or (step + 1) == total_batches:
                    curr_lr = optimizer.param_groups[0]['lr']
                    print(f"Step {global_step}/{total_steps} | Loss: {accum_loss:.4f} | LR: {curr_lr:.2e} | Time: {dt*1000:.1f}ms")
                accum_loss = 0.0
                
            epoch_loss += loss.item() * grad_accum_steps
        
        avg_loss = epoch_loss / total_batches
        print(f"Epoch {epoch+1} avg loss: {avg_loss:.4f}")
    
    # Save the final unified checkpoint
    os.makedirs("checkpoints", exist_ok=True)
    final_path = "checkpoints/megha_final.pt"
    torch.save(model.state_dict(), final_path)
    
    # Also save as level_14 for backwards compatibility with evaluate.py fallback
    torch.save(model.state_dict(), "checkpoints/megha_level_14.pt")
    
    print(f"\n{'='*50}")
    print(f"Training complete! Final model saved to {final_path}")
    print(f"Total steps trained: {global_step}")
    print(f"{'='*50}")


# Keep old function for backward compatibility
def train_level(level: int):
    """Deprecated: use train_all() instead."""
    print(f"Note: train_level() is deprecated. Use train_all() for better results.")
    train_all()


if __name__ == "__main__":
    train_all()



In [ ]:
%%writefile megha/evaluate.py
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from .model import MeghaModel
from .config import MeghaConfig
from .tokenizer import MeghaTokenizer
import os
import re

def run_evaluation():
    print("Starting MEGHA Evaluation Phase...")
    
    # Load MEGHA
    config = MeghaConfig()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    megha_model = MeghaModel(config).to(device)
    
    # Load the best available checkpoint (newest first)
    loaded = False
    for ckpt_name in ["checkpoints/megha_final.pt"] + \
                      [f"checkpoints/megha_level_{lvl}.pt" for lvl in range(14, -1, -1)]:
        if os.path.exists(ckpt_name):
            megha_model.load_state_dict(torch.load(ckpt_name, map_location=device))
            print(f"Loaded MEGHA from {ckpt_name}")
            loaded = True
            break
    if not loaded:
        print("CRITICAL: No checkpoint found! Evaluation will use random weights.")
        
    megha_model.eval()
    megha_tok = MeghaTokenizer(config)
    megha_tok.load("data/tokenizer.json")
    eos_id = megha_tok.get_eos_token_id()
    print(f"EOS token ID: {eos_id}")
    
    # Load Qwen (Teacher/Grader)
    print("Loading Teacher (Qwen 3B) for grading...")
    teacher_id = "Qwen/Qwen2.5-3B-Instruct"
    teacher_tok = AutoTokenizer.from_pretrained(teacher_id)
    teacher = AutoModelForCausalLM.from_pretrained(
        teacher_id, 
        torch_dtype=torch.float16, 
        device_map="auto"
    )
    
    # 10 test questions covering key curriculum levels
    test_questions = {
        "Level 1 (OS Basics)":      "What is the primary role of an operating system?",
        "Level 3 (Linux)":         "What is the command to change file permissions in Linux?",
        "Level 4 (Networking)":    "What is the purpose of DNS in computer networking?",
        "Level 5 (Databases)":     "What is the difference between a primary key and a foreign key?",
        "Level 6 (AWS)":            "What is Amazon EC2 used for?",
        "Level 7 (Docker)":         "What does a Dockerfile do?",
        "Level 8 (Kubernetes)":     "What is a Kubernetes Pod?",
        "Level 10 (Security)":      "Why should you not store AWS access keys in a public S3 bucket?",
        "Level 11 (Troubleshooting)":"If a website returns a 502 error, what could be the problem?",
        "Level 14 (CloudOps Incidents)":"How do you approach a multi-step CloudOps incident investigation?"
    }
    
    results = {}
    
    for topic, question in test_questions.items():
        print(f"\n[Testing {topic}]")
        print(f"Question: {question}")
        
        # ── 1. MEGHA generates an answer ────────────────────────────
        prompt = f"Q: {question}\nA:"
        input_ids = megha_tok.encode(prompt)
        if not input_ids:
            input_ids = [0]
            
        x = torch.tensor([input_ids], dtype=torch.long).to(device)
        
        with torch.no_grad():
            out_ids = megha_model.generate(
                x, max_new_tokens=80, temperature=0.35, top_k=40,
                eos_token_id=eos_id, repetition_penalty=1.25
            )
        
        # Decode ONLY the newly generated tokens (not the prompt)
        prompt_len = len(input_ids)
        new_token_ids = out_ids[0][prompt_len:].tolist()
        
        # Remove EOS token from the end if present
        if eos_id is not None and new_token_ids and new_token_ids[-1] == eos_id:
            new_token_ids = new_token_ids[:-1]
        
        megha_answer = megha_tok.decode(new_token_ids).strip()
        
        # Secondary cleanup: strip any repeated question text or A: prefix
        # (handles Whitespace tokenizer spacing quirks)
        for junk in ["A :", "A:", question]:
            if megha_answer.startswith(junk):
                megha_answer = megha_answer[len(junk):].strip()
        
        # Cut off if a new question starts
        for stop in ["Q :", "\nQ:", " Q:"]:
            if stop in megha_answer:
                megha_answer = megha_answer.split(stop)[0].strip()
        
        # Collapse multiple spaces from Whitespace tokenizer decode
        megha_answer = re.sub(r'\s+', ' ', megha_answer).strip()
        
        if not megha_answer:
            megha_answer = "[No answer generated]"
            
        print(f"MEGHA's Answer: {megha_answer}")
        
        # ── 2. Qwen grades the answer ────────────────────────────────
        grade_prompt = f"""You are grading an AI student's answer. Be generous — award partial marks for any relevant keywords or concepts.

Question: {question}

Student's Answer: {megha_answer}

Scoring guide:
- 0: Completely wrong, irrelevant, or gibberish
- 20-40: Mentions 1-2 relevant keywords but mostly incorrect
- 50-70: Partially correct, gets the main concept
- 80-100: Correct and complete answer

Output ONLY a single integer score between 0 and 100."""
        
        messages = [
            {"role": "system", "content": "You are a fair grader. Be generous with partial credit. Output only a number."},
            {"role": "user", "content": grade_prompt}
        ]
        
        text = teacher_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = teacher_tok([text], return_tensors="pt").to(teacher.device)
        
        with torch.no_grad():
            gen_ids = teacher.generate(
                **model_inputs,
                max_new_tokens=5,
                do_sample=False       # greedy for consistent scoring
            )
        
        new_ids = [out[len(inp):] for inp, out in zip(model_inputs.input_ids, gen_ids)]
        score_text = teacher_tok.batch_decode(new_ids, skip_special_tokens=True)[0].strip()
        
        # Extract first number found
        nums = re.findall(r'\d+', score_text)
        score = str(min(int(nums[0]), 100)) if nums else "0"
            
        print(f"Teacher's Grade: {score}/100")
        results[topic] = score
        
    print("\n" + "="*40)
    print("MEGHA FINAL REPORT CARD")
    print("="*40)
    total = 0
    for topic, score in results.items():
        print(f"{topic}: {score}%")
        total += int(score)
    avg = total // len(results)
    print(f"{'='*40}")
    print(f"Overall Average: {avg}%")
    print("="*40)

if __name__ == "__main__":
    run_evaluation()



In [ ]:
!python megha/data_gen.py --level 0 --real
!python megha/data_gen.py --level 1 --real
!python megha/data_gen.py --level 2 --real
!python megha/data_gen.py --level 3 --real
!python megha/data_gen.py --level 4 --real
!python megha/data_gen.py --level 5 --real
!python megha/data_gen.py --level 6 --real
!python megha/data_gen.py --level 7 --real
!python megha/data_gen.py --level 8 --real
!python megha/data_gen.py --level 9 --real
!python megha/data_gen.py --level 10 --real
!python megha/data_gen.py --level 11 --real
!python megha/data_gen.py --level 12 --real
!python megha/data_gen.py --level 13 --real
!python megha/data_gen.py --level 14 --real
!python -m megha.tokenizer
!python -c "from megha.train import train_all; train_all()"
!python -m megha.evaluate
!cp -r checkpoints/* /kaggle/working/ || true
!cp -r data /kaggle/working/ || true
